# V034: input-only physics residual matching

Select audited training controllers by how well they explain known inputs and baseline predictions. Compare two fixed correction caps. No query truth enters prediction.


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,time,inspect,platform
import numpy as np
from threadpoolctl import threadpool_limits
V034_RECIPE=dict(neighbors=5,passes=4,minimum_query_eligible=14,controller_selection='minimum_mean_squared_one_step_pressure_equation_residual_against_baseline_prediction',candidates=['physics_match_one_bin','physics_match_full_grid'],bootstrap_seed=20263401,bootstrap_draws=1000,minimum_gain=.003,threads=1)

def c34_code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
def c34_sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def c34_need(c,m):
 if not bool(c):raise ValueError(m)
def c34_json(p,v):
 with Path(p).open('x') as f:f.write(json.dumps(v,indent=2,allow_nan=False)+'\n')
def c34_namespace(root):
 p=Path(root)/'research/controller_bank_components_v033r1.ipynb';source=c34_code(p);c34_need(hashlib.sha256(source.encode()).hexdigest()=='2d9494051b42e59afa8c149eb8c5c3c00b0a49b795bba0193b378dd67d4c5fed','Pinned library component changed');ns={};exec(compile(source,str(p),'exec'),ns);return ns

def c34_match(inputs,rc,baseline,model,ns):
 """Score library parameters on known controls + model PREDICTIONS, not query truth."""
 _,distance,confidence=ns['c33_retrieve'](inputs,rc,model);u=inputs[:,:,0];phase=inputs[:,:,2];valid=np.zeros_like(baseline,dtype=bool)
 valid[:,2:]=(phase[:,2:]==0)&(phase[:,1:-1]==0)&(u[:,2:]>1e-6)&(u[:,2:]<99.999999)&(u[:,1:-1]>1e-6)&(u[:,1:-1]<99.999999)
 chosen=np.full((len(inputs),5),-1,dtype=np.int64);rmse=np.full((len(inputs),5),np.inf);usable=confidence&(valid.sum(1)>=14)
 beta=model['coef'];theta=np.column_stack([beta[:,0]/beta[:,1],beta[:,2]/beta[:,1],beta[:,3]/beta[:,1],1./beta[:,1]])
 for i in np.flatnonzero(usable):
  key=ns['c33_key'](rc[i],phase[i]);ii=model['groups'][key];k=np.flatnonzero(valid[i]);X=np.column_stack([u[i,k-1],baseline[i,k-1],np.ones(len(k)),-u[i,k]])
  residual=theta[ii]@X.T-baseline[i,k];score=np.mean(residual**2,axis=1);order=np.argsort(score,kind='stable')[:5];chosen[i]=ii[order];rmse[i]=np.sqrt(score[order])
 return chosen,rmse,usable,distance

def c34_predict(inputs,rc,baseline,model,grid,ns):
 """Two prospectively declared cap variants; query pressure is never accepted."""
 inputs=np.asarray(inputs,dtype=np.float64);baseline=np.asarray(baseline,dtype=np.float64)
 c34_need(inputs.shape==(len(rc),80,3) and baseline.shape==(len(rc),80) and np.isfinite(inputs).all() and np.isfinite(baseline).all(),'Input-only query shape/finiteness')
 nn,rmse,usable,distance=c34_match(inputs,rc,baseline,model,ns);curves,valid=ns['c33_project'](inputs,baseline,model['coef'][np.maximum(nn,0)],4)
 proposal=np.median(curves,axis=1);iqr=np.quantile(curves,.75,axis=1)-np.quantile(curves,.25,axis=1);gate=valid&usable[:,None]&(iqr<=model['lattice']['spacing'])
 bounded=ns['c33_one_bin'](baseline,proposal,gate,grid,model['lattice'])
 hi=np.searchsorted(grid,proposal).clip(0,len(grid)-1);lo=np.maximum(hi-1,0);nearest=np.where(abs(proposal-grid[lo])<=abs(proposal-grid[hi]),grid[lo],grid[hi]);full=np.where(gate,nearest,baseline)
 return dict(physics_match_one_bin=bounded,physics_match_full_grid=full),dict(neighbors=nn,equation_RMSE=rmse,usable=usable,known_input_distances=distance,proposal=proposal,iqr=iqr,gate=gate,valid=valid)

def c34_preflight(root):
 ns=c34_namespace(root);prior=ns['c33_preflight'](root);checks=list(prior['checks'])
 def need(c,m):c34_need(c,m);checks.append(m)
 n=8;x=np.zeros((n,80,3));x[:,:,0]=10+np.arange(n)[:,None]*.1;x[:,:,1]=np.arange(80)*.03;x[:,40:,2]=1;rc=np.tile([20,20],(n,1));model=ns['c33_index'](x,rc);model.update(coef=np.column_stack([np.full(n,.9),np.full(n,2.),np.ones(n),np.arange(n)*.1+1]),lattice=dict(origin=-20.,spacing=1.));baseline=np.full((1,80),5.);grid=np.arange(-20.,51.)
 pred,d=c34_predict(x[:1],rc[:1],baseline,model,grid,ns);k=np.flatnonzero(d['valid'][0]);scores=[]
 for b in model['coef']:
  dd,A,B,c=b;v=(dd*x[0,k-1,0]+B*baseline[0,k-1]+c-x[0,k,0])/A-baseline[0,k];scores.append(np.mean(v*v))
 need(np.array_equal(d['neighbors'][0],np.argsort(scores,kind='stable')[:5]),'Physics parameter selection independently matches native one-step equation residual')
 need(np.allclose(d['equation_RMSE'][0],np.sqrt(np.sort(scores)[:5])),'Selection RMSE native pressure units')
 need(set(pred)==set(V034_RECIPE['candidates']),'Exactly two predeclared cap variants')
 need(np.all(abs(pred['physics_match_one_bin']-baseline)<=1.),'Bounded variant preserves one-level movement cap')
 for name,p in pred.items():need(np.array_equal(p[~d['gate']],baseline[~d['gate']]),name+' fallback/boundaries exact')
 need('pressure' not in inspect.signature(c34_predict).parameters and 'truth' not in inspect.signature(c34_match).parameters,'New predictor and selector accept no query labels')
 xx=x[:1].copy();xx[:,:,0]=100;out,diag=c34_predict(xx,rc[:1],baseline,model,grid,ns);need(not diag['usable'][0] and all(np.array_equal(a,baseline) for a in out.values()),'Clipped or insufficient eligible query equations never select a controller')
 return dict(status='passed',check_count=len(checks),checks=checks,no_approved_data_access=True)

def run_controller_selection_v034(root,*,expected_protocol_sha256,expected_source_sha256):
 root=Path(root).resolve();proto=root/'research/controller_selection_protocol_v034.json';source=root/'research/controller_selection_components_v034.ipynb';c34_need(c34_sha(proto)==expected_protocol_sha256 and hashlib.sha256(c34_code(source).encode()).hexdigest()==expected_source_sha256,'Own frozen source/protocol changed');protocol=json.loads(proto.read_text());c34_need(protocol['recipe']==V034_RECIPE,'Fixed recipe changed')
 for rel,h in protocol['input_hashes'].items():c34_need(c34_sha(root/rel)==h,'Input changed: '+rel)
 o=root/'outputs/controller_selection_screen_v034';c34_need(not o.exists(),'Preserve completed/partial files');o.mkdir();start=time.perf_counter();ns=c34_namespace(root);c34_json(o/'preflight.json',c34_preflight(root));split=np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz');bank=root/'outputs/controller_bank_screen_v033r1';save=ns['c33_save'];c34_need(len(split['A_fit'])==61114 and len(split['A_validation'])==6791 and not np.intersect1d(split['A_fit'],split['A_validation']).size,'Original A groups')
 print('Reconstructing audited training-only bank index; no controller refit',flush=True)
 with threadpool_limits(limits=1):
  fb,fx,frc,frows,_=ns['c33_load'](root/'data/train.csv',split['A_fit'],pressure=False);idx=np.load(bank/'bank_indices.npy');model=ns['c33_index'](fx[idx],frc[idx]);model.update(coef=np.load(bank/'bank_coef.npy'),lattice=json.loads((bank/'lattice.json').read_text()))
  c34_need(np.array_equal(fb[idx],np.load(bank/'bank_breath_ids.npy')),'Training controller bank ID identity')
  for name in ['mean','std','loo_fifth']:c34_need(np.array_equal(model[name],np.load(bank/('bank_'+name+'.npy')),equal_nan=True),'Bank controls-only statistics changed')
  vb,vx,vrc,vr,_=ns['c33_load'](root/'data/train.csv',split['A_validation'],pressure=False);order=np.argsort(vr[:,0],kind='stable');vb,vx,vrc,vr=vb[order],vx[order],vrc[order],vr[order];mask=vx[:,:,2]==0
  c34_need(np.array_equal(vr[mask],np.load(root/'outputs/controlled_experiments_v013/A_row_ids.npy')) and mask.sum()==206098,'Canonical cached A prediction alignment')
  baseline=np.zeros((len(vb),80));baseline[mask]=np.load(root/'outputs/postprocessing_screen_v016/weighted_median_midpoint_then_nearest_training_pressure_codebook_A_predictions.npy');grid=np.load(bank/'training_pressure_codebook.npy')
  print('Selecting controllers using predicted pressure + known controls only',flush=True);predictions,diagnostics=c34_predict(vx,vrc,baseline,model,grid,ns)
  for name,p in predictions.items():save(o/(name+'_A_predictions.npy'),p[mask])
  for name,a in diagnostics.items():save(o/('A_'+name+'.npy'),a)
  save(o/'A_row_ids.npy',vr[mask]);save(o/'A_breath_ids.npy',vb)
  c34_json(o/'predictions_locked_before_labels.json',dict(created_utc=datetime.now(timezone.utc).isoformat(),protocol_sha256=expected_protocol_sha256,source_code_sha256=expected_source_sha256,query_pressure_absent=True,prediction_sha256={name:c34_sha(o/(name+'_A_predictions.npy')) for name in predictions}))
  print('Both candidate predictions locked; loading pressure labels for scoring only',flush=True)
  tb,tx,trc,tid,truth=ns['c33_load'](root/'data/train.csv',split['A_validation'],pressure=True);tb,tx,tid,truth=tb[order],tx[order],tid[order],truth[order];c34_need(np.array_equal(tb,vb) and np.array_equal(tx,vx) and np.array_equal(tid,vr),'Truth alignment');truth=truth.astype(np.float32).astype(np.float64);be=abs(baseline-truth);ref=float(be[mask].mean());c34_need(abs(ref-.18292209456473527)<1e-12,'Accepted A baseline mismatch');rows=[]
  import pandas as pd
  rcrows=[]
  for name,p in predictions.items():
   e=abs(p-truth);mae=float(e[mask].mean());gain=ref-mae;interval,draws=ns['c33_bootstrap'](p,baseline,truth,vx[:,:,2],seed=20263401);save(o/(name+'_bootstrap_deltas.npy'),draws);changed=(p!=baseline)&mask
   rows.append(dict(name=name,MAE=mae,median_absolute_error=float(np.median(e[mask])),gain=gain,bootstrap_95=interval,eligible=bool(gain>=.003 and interval[1]<0),changed_points=int(changed.sum()),changed_breaths=int(changed.any(1).sum()),improved_changed_points=int((changed&(e<be)).sum()),worsened_changed_points=int((changed&(e>be)).sum()),prediction_sha256=c34_sha(o/(name+'_A_predictions.npy'))))
   for setting in sorted(set(map(tuple,vrc))):
    use=np.all(vrc==setting,axis=1);m=mask[use];err=e[use][m];old=be[use][m];rcrows.append(dict(candidate=name,R=int(setting[0]),C=int(setting[1]),scored_points=int(m.sum()),MAE=float(err.mean()),baseline_MAE=float(old.mean()),total_absolute_error=float(err.sum()),delta_total_absolute_error=float(err.sum()-old.sum())))
  pd.DataFrame(rcrows).to_csv(o/'RC_errors.csv',index=False);eligible=[r for r in rows if r['eligible']];selected=min(eligible,key=lambda a:(a['MAE'],a['name']))['name'] if eligible else None
  result=dict(status='complete',completed_utc=datetime.now(timezone.utc).isoformat(),protocol_sha256=expected_protocol_sha256,source_code_sha256=expected_source_sha256,recipe=V034_RECIPE,reference_A_MAE=ref,candidates=rows,selected_for_confirmation=selected,bank_size=len(idx),validation_breaths=len(vb),validation_scored_points=int(mask.sum()),usable_breaths=int(diagnostics['usable'].sum()),gate_points=int((diagnostics['gate']&mask).sum()),runtime_seconds=time.perf_counter()-start,reused_training_only_bank=True,gradient_model_fits=0,neural_checkpoint_inferences=0,no_GPU=True,no_test_data=True,no_F_split_or_fit=True,no_final_fit_or_submission=True,query_pressure_absent_from_prediction=True,runtime=dict(python=platform.python_version(),numpy=np.__version__),artifact_sha256={str(p.relative_to(o)):c34_sha(p) for p in sorted(o.iterdir()) if p.is_file()})
  c34_json(o/'summary.json',result);return result
